In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

db = duckdb.connect(
    "C:/Users/Decss/Desktop/retail-transaction-analyses/Data/retail.db",
    read_only=True
)
db.sql("SET search_path TO mart;")
db.sql("""SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'mart'""")

┌─────────────────┐
│   table_name    │
│     varchar     │
├─────────────────┤
│ customerdim     │
│ productdim      │
│ retail_enriched │
└─────────────────┘

# Product Performance

Vilka produkter driver verksamhetens intäkter — och gör det effektivt?

- Var är försäljningen koncentrerad, och hur fragmenterad är katalogen?
- Vilka produkter uppvisar oroväckande returmönster som urholkar intäkterna?
- Finns det produkter med outnyttjad potential som når för smalt?


# Notering:

Produktanomalier filtreras bort för att datan bättre ska spegla kärnverksamheten.

"AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)" 

In [2]:
# Total produktförsäljning (is_product = 1, inkl. anomalikunder via productdim)
db.sql("""SELECT 
       SUM(total_sales) 
       FROM productdim 
       WHERE is_product = 1
       """)

┌────────────────────┐
│  sum(total_sales)  │
│       double       │
├────────────────────┤
│ 10261631.034000006 │
└────────────────────┘

In [3]:
# Antal aktiva produkter i katalogen
db.sql("SELECT COUNT(DISTINCT stockcode) as active_items FROM productdim WHERE is_product = 1 AND total_sales IS NOT NULL")

┌──────────────┐
│ active_items │
│    int64     │
├──────────────┤
│         3910 │
└──────────────┘

In [4]:
# Genomsnittlig returgrad och returvärde per produkt
df_avg_rr = db.sql("""
    SELECT
    ROUND(SUM(return_rate_order * orders_with_product) / SUM(orders_with_product), 4) as weighted_return_rate_order,
    SUM(return_rate_value * total_sales) / SUM(total_sales) as weighted_return_rate_value
    FROM productdim
    WHERE total_sales IS NOT NULL
    AND is_product = 1
    AND stockcode NOT IN (
        SELECT DISTINCT stockcode
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
""").df()
print(df_avg_rr)

   weighted_return_rate_order  weighted_return_rate_value
0                      0.0159                    0.024078


Den totala försäljningen för datasettets period (2010-12-01 08:26:00 till 2011-12-09 12:50:00) uppgår till £10261631.8

Antal unika och aktivt handlade produkter i katalogen uppgår till 3910 stycken.

Den viktade returgraden av en produkt är 1,59% med genomsnittligt returvärde på 2,4%.

In [5]:
# Top products by total sales
df_top_product = db.sql("""
       SELECT 
       stockcode,
       description,
       ROUND(total_sales, 2) as total_sales,
       orders_with_product 
       FROM 
       productdim
       WHERE total_sales IS NOT NULL
       AND is_product = 1
       AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)
       ORDER BY total_sales desc                       
       """).df()
print(df_top_product.head(50))

   StockCode                          description  total_sales  \
0      22423             REGENCY CAKESTAND 3 TIER    174156.54   
1     85123A   WHITE HANGING HEART T-LIGHT HOLDER    104462.75   
2      47566                        PARTY BUNTING     99445.23   
3     85099B              JUMBO BAG RED RETROSPOT     94159.81   
4      23084                   RABBIT NIGHT LIGHT     66870.03   
5      22086      PAPER CHAIN KIT 50'S CHRISTMAS      64875.59   
6      84879        ASSORTED COLOUR BIRD ORNAMENT     58927.62   
7      79321                        CHILLI LIGHTS     54096.36   
8      22197                       POPCORN HOLDER     51334.47   
9      23298                       SPOTTY BUNTING     43148.18   
10     22386              JUMBO BAG PINK POLKADOT     42401.01   
11     23203             JUMBO BAG VINTAGE DOILY      42030.34   
12     21137             BLACK RECORD COVER FRAME     40633.38   
13     23284        DOORMAT KEEP CALM AND COME IN     38133.64   
14     227

In [6]:
# Product sale frequency dist
px.histogram(df_top_product,
             x='total_sales',
             nbins=250,
             title='Product Sales Distribution'
             )

In [7]:
# Top 20 products by sale
px.bar(
    df_top_product.head(25),
    x='description',
    y='total_sales',
    title='Top 25 items by total sales'
)

I grafen "top 20 products by sale" ser vi de 20 produkter som bidrar mest till totala intäkterna. Det är däremot viktigt att poängtera hur intäktssiffrorna per produkt relaterar till totala intäkter.

Som enskild produkt så sticker "REGENCY CAKESTAND 3 TIER" ut baserat på total_sales, men 174,156.54 är trotts allt inte mer än 1,697% av totala försäljningen. När vi sedan tittar på produkter som rankas lägre strax därefter i total_sales så ser vi att total_sales på ensiklda produkter avtar ganska snabbt (1. £174,156.54 -> 20. £33,146.82). Försäljningen är alltså i stort sett ganska fragmenterad, och det är inte ett fåtal produkter som står för majoriteten av intäkterna.

Distributionen i grafen "product sale frequency dist" indikerar att de flesta produkter inte säljer för mer än £0-2500.

Bland våra "top products by total sales" så kan vi se att dess förekomst inom orders varierar mellan totalt ~375 och ~2203 gånger. En del av dessa varor är alltså högt upp på denna lista för att de faktiskt ofta förekommer i beställningar.

Men vi behöver sätta dessa värden i relation till hela datasettet för att avgöra vilken grad av frekvent 2000 gånger faktiskt är.

In [8]:
# Top frequent products
df_top_frequent = db.sql("""
       SELECT 
       stockcode, 
       description, 
       orders_with_product as in_orders, 
       ROUND(total_sales, 2) as total_sales 
       FROM productdim 
       WHERE is_product = 1 
       AND total_sales IS NOT NULL
       AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)
       ORDER BY orders_with_product desc
       """).df()
print(df_top_frequent.head(25))

   StockCode                         description  in_orders  total_sales
0     85123A  WHITE HANGING HEART T-LIGHT HOLDER       2203    104462.75
1     85099B             JUMBO BAG RED RETROSPOT       2092     94159.81
2      22423            REGENCY CAKESTAND 3 TIER       1989    174156.54
3      47566                       PARTY BUNTING       1686     99445.23
4      20725             LUNCH BAG RED RETROSPOT       1565     35862.36
5      84879       ASSORTED COLOUR BIRD ORNAMENT       1455     58927.62
6      22197                      POPCORN HOLDER       1392     51334.47
7      22720   SET OF 3 CAKE TINS PANTRY DESIGN        1387     38108.89
8      21212     PACK OF 72 RETROSPOT CAKE CASES       1320     21246.45
9      22383              LUNCH BAG SUKI DESIGN        1285     22486.34
10     20727             LUNCH BAG  BLACK SKULL.       1273     22346.96
11     22457     NATURAL SLATE HEART CHALKBOARD        1249     27991.61
12     23203            JUMBO BAG VINTAGE DOILY    

In [9]:
# 
fig_freq_sale = make_subplots(specs=[[{"secondary_y": True}]])

fig_freq_sale.add_trace(
    go.Bar(x=df_top_frequent.head(25)['description'],
           y=df_top_frequent.head(25)['in_orders'],
           name='Orders'),
           secondary_y=False
)
fig_freq_sale.add_trace(
    go.Scatter(x=df_top_frequent.head(25)['description'],
           y=df_top_frequent.head(25)['total_sales'],
           name='Total Sales',
           mode='lines+markers'),
           secondary_y=True
)
fig_freq_sale.update_layout(
    title_text="Order frequency and total sales ranked by order frequency"
)
fig_freq_sale.update_xaxes(
    title_text="Item description"
)
fig_freq_sale.update_yaxes(
    title_text="Order count", secondary_y=False,)
fig_freq_sale.update_yaxes(
   title_text="Total Sales", secondary_y=True
)

När vi sorterat på de mest frekvent förekommande produkterna så kan vi se att det finns en överlappning av de produkter som har en hög total_sales och ofta förekommer i orders. Sambandet är däremot inte linjärt och belyser samspelet mellan både pris och frekvens i produktens inverkan på total försäljning.

In [10]:
# Top 50 by total_sale share of sales
share_top50_sales = df_top_product.head(50)['total_sales'].sum() / df_top_product['total_sales'].sum()
print(share_top50_sales.round(3))

0.204


In [11]:
# Top 50 by order frequency share of sales
share_top50_freq = df_top_frequent.head(50)['total_sales'].sum() / df_top_frequent['total_sales'].sum()
print(share_top50_freq.round(3))

0.181


Av de totalt 3911 aktiva produkter i våran katalog så står top 50 för 20,4% av försäljningen när man rankar på försäljning och 18,1%  rankat på frekvens i orders. Dessa produkter är alltså 1,28% av vår totala produktkatalog. Trotts att ca 20% av intäkterna är relativt koncentrerat till endast 50 produkter så tyder det också på att en stor del av resterande intäkter kommer från den breda produktkatalogen. Verksamheten är på så sätt inte beroende av ett fåtal produkter.

Hur ser då returgraden ut bland dessa produkter? Finns det produkter som sticker ut?

In [12]:
# Return rate
# Top products by total sales
df_return_top_sale = db.sql("""
       SELECT 
       stockcode,
       description,
       ROUND(total_sales, 2) as total_sales,
       return_rate_order 
       FROM 
       productdim
       WHERE total_sales IS NOT NULL
       AND is_product = 1
       AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)
       ORDER BY total_sales desc                       
       """).df()
print(df_return_top_sale.head(25)[['StockCode','description', 'total_sales', 'return_rate_order']])

   StockCode                         description  total_sales  \
0      22423            REGENCY CAKESTAND 3 TIER    174156.54   
1     85123A  WHITE HANGING HEART T-LIGHT HOLDER    104462.75   
2      47566                       PARTY BUNTING     99445.23   
3     85099B             JUMBO BAG RED RETROSPOT     94159.81   
4      23084                  RABBIT NIGHT LIGHT     66870.03   
5      22086     PAPER CHAIN KIT 50'S CHRISTMAS      64875.59   
6      84879       ASSORTED COLOUR BIRD ORNAMENT     58927.62   
7      79321                       CHILLI LIGHTS     54096.36   
8      22197                      POPCORN HOLDER     51334.47   
9      23298                      SPOTTY BUNTING     43148.18   
10     22386             JUMBO BAG PINK POLKADOT     42401.01   
11     23203            JUMBO BAG VINTAGE DOILY      42030.34   
12     21137            BLACK RECORD COVER FRAME     40633.38   
13     23284       DOORMAT KEEP CALM AND COME IN     38133.64   
14     22720   SET OF 3 C

Vi zoomar in på de 25 produkter som ligger i toppen (sale, frekvens) och kikar på hur ofta en retur sker där produkten förekommer i en order. De flesta av våra produkter som presterar bra har en returgrad som är nära eller under genomsnittet bland produkter som returneras. Detta bör tolkas positivt då kunder som beställer till stor del får sina förväntningar mötta. I de fall där vi kan se att returgraden är en bit över genomsnittet, men ändå från en produkt som är populär bör vi undersöka varför returer sker och be om feedback från våra kunder i dessa fall.

In [13]:
# Return rate
# Top frequent products
df_return_top_frequent = db.sql("""
       SELECT 
       stockcode, 
       description, 
       orders_with_product as in_orders, 
       ROUND(total_sales, 2) as total_sales,
       return_rate_order 
       FROM productdim 
       WHERE is_product = 1 
       AND total_sales IS NOT NULL
       AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)
       ORDER BY orders_with_product desc
       """).df()
print(df_return_top_frequent.head(25)[['StockCode', 'in_orders', 'return_rate_order']])

   StockCode  in_orders  return_rate_order
0     85123A       2203           0.018700
1     85099B       2092           0.020141
2      22423       1989           0.082873
3      47566       1686           0.011723
4      20725       1565           0.026741
5      84879       1455           0.008174
6      22197       1392           0.034674
7      22720       1387           0.049932
8      21212       1320           0.010495
9      22383       1285           0.016080
10     20727       1273           0.016988
11     22457       1249           0.013428
12     23203       1232           0.013611
13     22386       1218           0.010561
14     22469       1202           0.009061
15     21931       1184           0.014155
16     22411       1175           0.010110
17     22961       1162           0.010221
18     22086       1160           0.008547
19     22382       1157           0.011111
20     20728       1150           0.019608
21     23298       1147           0.010354
22     2296

In [14]:
r_topsale = df_return_top_sale.head(25).copy()
r_topsale['color'] = r_topsale['return_rate_order'].apply(lambda x: 'Över Snitt' if x > df_avg_rr['weighted_return_rate_order'][0] else 'Under Snitt')

fig_r_topsale = px.bar(r_topsale,
                       x='description',
                       y='return_rate_order',
                       color='color',
                       color_discrete_map={'Över Snitt': 'crimson', 'Under Snitt': 'steelblue'},
                       title='Return Rate Top product (sale)',
                       category_orders={'description': r_topsale['description'].tolist()}
                       )

fig_r_topsale.add_hline(
    y=df_avg_rr['weighted_return_rate_order'][0],
    line_dash='dash',
    line_color='red'    
)

fig_r_topsale.update_yaxes(tickformat='.2%')

fig_r_topsale.show()

Bland top25-produkterna (sälj) så kan vi se att det är 5st varor som sticker ut från medelvärdet, varav en är vår bäst säljande produkt. Anledning till retur av dessa produkter bör tittas närmare på.

In [15]:
r_topfreq = df_return_top_frequent.head(25).copy()
r_topfreq['color'] = r_topfreq['return_rate_order'].apply(lambda x: 'Över Snitt' if x > df_avg_rr['weighted_return_rate_order'][0] else 'Under Snitt')

fig_r_topfreq = px.bar(r_topfreq,
                       x='description',
                       y='return_rate_order',
                       color='color',
                       color_discrete_map={'Över Snitt': 'crimson', 'Under Snitt': 'steelblue'},
                       title='Return Rate Top product (frequency)',
                       category_orders={'description': r_topfreq['description'].tolist()}
                       )

fig_r_topfreq.add_hline(
    y=df_avg_rr['weighted_return_rate_order'][0],
    line_dash='dash',
    line_color='red'    
)

fig_r_topfreq.update_yaxes(tickformat='.2%')

fig_r_topfreq.show()

Vi har förvisso en del överlappande produkter bland de som rankas på frekvens. Bland de som sticker ut över genomsnittlig return rate per order så skiljer sig en. Samma resonemang här kring prestation och plan of action (anledning saknas i dataset).

Returer sker ju av olika anledningar. Det kan vara allt från felbeställningar, skador, upplevda fel eller defferenser från förväntning, lagerproblem mm. 

För att förstå dessa returer något bättre så ska vi titta närmare på hur stor del av intäkterna som returneras. Detta skulle kunna ge en hint om vad för typ av returer vi hanterar per produkt.

In [16]:
# Return value rate
# Top products by total sales
df_rvalue_top_sale = db.sql("""
       SELECT 
       stockcode,
       description,
       ROUND(total_sales, 2) as total_sales,
       return_rate_value
       FROM 
       productdim
       WHERE total_sales IS NOT NULL
       AND is_product = 1
       AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)
       ORDER BY total_sales desc                       
       """).df()
print(df_rvalue_top_sale.head(25)[['StockCode','description', 'total_sales', 'return_rate_value']])

   StockCode                         description  total_sales  \
0      22423            REGENCY CAKESTAND 3 TIER    174156.54   
1     85123A  WHITE HANGING HEART T-LIGHT HOLDER    104462.75   
2      47566                       PARTY BUNTING     99445.23   
3     85099B             JUMBO BAG RED RETROSPOT     94159.81   
4      23084                  RABBIT NIGHT LIGHT     66870.03   
5      22086     PAPER CHAIN KIT 50'S CHRISTMAS      64875.59   
6      84879       ASSORTED COLOUR BIRD ORNAMENT     58927.62   
7      79321                       CHILLI LIGHTS     54096.36   
8      22197                      POPCORN HOLDER     51334.47   
9      23298                      SPOTTY BUNTING     43148.18   
10     22386             JUMBO BAG PINK POLKADOT     42401.01   
11     23203            JUMBO BAG VINTAGE DOILY      42030.34   
12     21137            BLACK RECORD COVER FRAME     40633.38   
13     23284       DOORMAT KEEP CALM AND COME IN     38133.64   
14     22720   SET OF 3 C

In [17]:
#
rv_topsale = df_rvalue_top_sale.head(25).copy()
rv_topsale['color'] = rv_topsale['return_rate_value'].apply(lambda x: 'Över Snitt' if x > df_avg_rr['weighted_return_rate_value'][0] else 'Under Snitt')

fig_rv_topsale = px.bar(rv_topsale,
                       x='description',
                       y='return_rate_value',
                       color='color',
                       color_discrete_map={'Över Snitt': 'crimson', 'Under Snitt': 'steelblue'},
                       title='Return value rate Top products (sale)',
                       category_orders={'description': rv_topsale['description'].tolist()}
                       )

fig_rv_topsale.add_hline(
    y=df_avg_rr['weighted_return_rate_value'][0],
    line_dash='dash',
    line_color='red'    
)

fig_rv_topsale.update_yaxes(tickformat='.2%')

fig_rv_topsale.show()

Det viktade returvärdet är 2,4% av intäkterna. I grafen ovan, som representerar våra bäst säljande produkter, ser vi att bortsett från tre stycken så har de flesta ett lägre returvärde än det viktade värdet. Produkter som säljer bra är alltså varken mer sannolika att returneras, och värdet som returneras är procentuellt inte högre än de viktade genomsnitten. Med ett undantag — Regency Cakestand 3 Tier — som vi återkommer till nedan.

In [18]:
# Order penetration top 25 (sales)
df_orderpen = db.sql("""
                     SELECT
                     stockcode,
                     description,
                     total_sales,
                     orders_with_product,
                     order_penetration
                     
                     FROM productdim
                     WHERE is_product = 1
                     AND stockcode NOT IN (
                        SELECT DISTINCT stockcode
                        FROM staging.top_anomalies
                        WHERE anomaly_type = 'anomaly_customer'
                     )
                     ORDER BY total_sales desc
""").df()
print(df_orderpen[['description','order_penetration', 'orders_with_product']].head(25))
df_orderpen[['order_penetration', 'orders_with_product']].describe()

                           description  order_penetration  orders_with_product
0             REGENCY CAKESTAND 3 TIER           0.095957                 1989
1   WHITE HANGING HEART T-LIGHT HOLDER           0.106281                 2203
2                        PARTY BUNTING           0.081339                 1686
3              JUMBO BAG RED RETROSPOT           0.100926                 2092
4                   RABBIT NIGHT LIGHT           0.048582                 1007
5      PAPER CHAIN KIT 50'S CHRISTMAS            0.055963                 1160
6        ASSORTED COLOUR BIRD ORNAMENT           0.070195                 1455
7                        CHILLI LIGHTS           0.032179                  667
8                       POPCORN HOLDER           0.067156                 1392
9                       SPOTTY BUNTING           0.055336                 1147
10             JUMBO BAG PINK POLKADOT           0.058761                 1218
11            JUMBO BAG VINTAGE DOILY            0.0

,order_penetration,orders_with_product
count,4053.000000,4053.000000
mean,0.006154,127.558599
std,0.009235,191.425070
min,0.000000,0.000000
25%,0.000627,13.000000
50%,0.002846,59.000000
75%,0.007719,160.000000
max,0.106281,2203.000000



Bland de bäst säljande produkterna kan vi också se att order-penetration i de allra flesta fall är över medianen. Ett par av varorna är till och med de med högst order-penetration i hela katalogen. Detta stärker bilden av att top-produkterna säljer både brett och i hög volym, vilket gör dem centrala för verksamhetens totala intäktsgenerering.

Viktigt att belysa är produkten "Regency Cakestand 3 Tier" som avvikter över de viktade medelvärden i både returgrad (return_rate_order) 8,29% och returvärde (return_rate_value) 5,57%. I och med att denna produkt står för en signifik andel av försäljningen så tyder dessa siffror på att den också står för en betydande del av returer. Varför och vad man kan göra för att minska detta bör undersökas i separat analys.

Förutom "Regency Cakestand 3 Tier" bland top-produkterna så var det inte någon som riktigt stack ut på både returgrad och returvärde. Men det är mycket möjligt att vi har fler produkter i katalogen som faktiskt gör det.

Vi bör därav kika på den breda produktkatalogen efter produkter som uppvisar liknande beteende som "Regency Cakestand 3 Tier". Där exempelvis returgrad är hög eller där totala returvärdet är högt.

Samtidigt är det relevant att identifiera produkter som har låg returgrad i förhållande till försäljning, samt produkter med låg order-penetration. Denna skapar en grund för att segmentera produkter och därefter hantera dem utifrån deras faktiska beteende.

In [19]:
# Översikt av värden vi definierar segment med
df_scout = db.sql("""
                  SELECT
                  *
                  FROM productdim
                  WHERE is_product = 1
                  And total_sales IS NOT NULL
                  AND stockcode NOT IN (
                  SELECT DISTINCT stockcode
                  FROM mart.retail_enriched
                  WHERE customerid IN (
                  SELECT DISTINCT customerid
                  FROM staging.top_anomalies
                  WHERE anomaly_type = 'anomaly_customer')
                  )
                  """).df()
df_scout[['total_sales', 'orders_with_product', 'return_rate_quantity', 'return_rate_value', 'order_penetration']].describe()

,total_sales,orders_with_product,return_rate_quantity,return_rate_value,order_penetration
count,3905.000000,3905.000000,1920.000000,1920.000000,3905.000000
mean,2549.192820,132.388220,0.068053,0.098780,0.006387
std,6402.139462,193.374704,0.714138,1.524675,0.009329
min,0.003000,1.000000,0.000123,0.000111,0.000048
25%,128.630000,16.000000,0.005761,0.005635,0.000772
50%,679.070000,65.000000,0.014632,0.014400,0.003136
75%,2213.420000,165.000000,0.035821,0.035430,0.007960
max,174156.540000,2203.000000,30.000000,49.210151,0.106281


Vi börjar med att definiera tre segment som ska identifiera produkter av olika karaktär.

* Stars
    - Detta är produkterna i vår katalog som driver intäkter effektivt. Produkter som är av vikt för verksamheten idag. De kännetecknas främst av kombinationen hög försäljning, låg returgrad och ofta hög penetration.

* Risky
    - Detta är produkter som fortfarande driver intäkter men inte lika effektivt som "Stars". Dessa produkter underpresterar av någon anledning på returmåtten för värde och/eller kvantitet.

* Niche
    - Nichade produkter är i detta fall sådana som säljer bra men inte når särskilt brett. Ofta handlar det om produkter som är lite dyrare, där försäljning är mindre frekvent, men när försäljning sker så genererar det bra värde.

Segmenteringen av produkter bidrar till ökad förståelse för produktkatalogen och hur de bidrar till intäkter och returer. Det hjälper oss också att bättre angripa eventuella problem kopplat till en produkt eller bättre utveckla produkter med potential.

In [20]:
# "Star" produkter 
df_stars = db.sql("""
       SELECT 
       stockcode, 
       description, 
       orders_with_product as in_orders,
       order_penetration, 
       ROUND(total_sales, 2) as total_sales,
       return_rate_quantity,
       return_rate_value 
       FROM productdim 
       WHERE is_product = 1 
       AND total_sales > 20000 
       AND total_sales IS NOT NULL
       AND return_rate_quantity < 0.01
       AND return_rate_value < 0.006  
       AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)
       ORDER BY return_rate_quantity asc, in_orders desc
       """).df()
print(df_stars.head(25)[['StockCode', 'description', 'in_orders', 'total_sales', 'return_rate_quantity', 'return_rate_value', 'order_penetration']])
df_stars[['return_rate_quantity']].describe()

   StockCode                          description  in_orders  total_sales  \
0      48194                       DOORMAT HEARTS        593     22114.42   
1      22189              CREAM HEART CARD HOLDER        524     23435.82   
2      48187                  DOORMAT NEW ENGLAND        600     24108.39   
3      21623         VINTAGE UNION JACK MEMOBOARD        221     22980.24   
4      21137             BLACK RECORD COVER FRAME        375     40633.38   
5      21523   DOORMAT FANCY FONT HOME SWEET HOME        502     21593.74   
6      84879        ASSORTED COLOUR BIRD ORNAMENT       1455     58927.62   
7      23209             LUNCH BAG VINTAGE DOILY        1099     21418.82   
8      21915               RED  HARMONICA IN BOX         662     26294.08   
9      48138                   DOORMAT UNION FLAG        616     25534.15   
10     23084                   RABBIT NIGHT LIGHT       1007     66870.03   
11     22629                  SPACEBOY LUNCH BOX         900     25719.02   

,return_rate_quantity
count,19.000000
mean,0.003396
std,0.002070
min,0.000878
25%,0.001414
50%,0.003199
75%,0.004772
max,0.008276


In [21]:
# "Risky" produkter 
df_risky = db.sql("""
       SELECT 
       stockcode, 
       description, 
       orders_with_product as in_orders, 
       ROUND(total_sales, 2) as total_sales,
       return_rate_quantity,
       return_rate_value           
       FROM productdim 
       WHERE is_product = 1 
       AND total_sales > 20000
       AND total_sales IS NOT NULL
       AND return_rate_quantity BETWEEN 0.04 AND 0.5
       AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)
       ORDER BY return_rate_quantity asc, in_orders desc
       """).df()
print(df_risky.head(25)[['StockCode', 'description', 'in_orders', 'total_sales', 'return_rate_quantity', 'return_rate_value']])

  StockCode                         description  in_orders  total_sales  \
0     23284       DOORMAT KEEP CALM AND COME IN        731     38133.64   
1     22699    ROSES REGENCY TEACUP AND SAUCER        1066     29056.92   
2     23201                  JUMBO BAG ALPHABET        906     27850.83   
3     23199                    JUMBO BAG APPLES        978     30450.69   
4     22423            REGENCY CAKESTAND 3 TIER       1989    174156.54   
5    85123A  WHITE HANGING HEART T-LIGHT HOLDER       2203    104462.75   
6     21843            RED RETROSPOT CAKE STAND        539     21354.30   
7     21175         GIN + TONIC DIET METAL SIGN        809     27029.97   
8     71477   COLOUR GLASS. STAR T-LIGHT HOLDER        320     20650.66   
9     48185                  DOORMAT FAIRY CAKE        365     23872.91   

   return_rate_quantity  return_rate_value  
0              0.040909           0.041990  
1              0.044298           0.039590  
2              0.044441           0.038

In [22]:
# "Niche" produkter 
df_niche = db.sql("""
       SELECT 
       stockcode, 
       description, 
       orders_with_product as in_orders,
       order_penetration, 
       ROUND(total_sales, 2) as total_sales,
       return_rate_quantity,
       return_rate_value 
       FROM productdim 
       WHERE is_product = 1 
       AND total_sales > 5000 
       AND total_sales IS NOT NULL
       AND order_penetration < 0.0031
       AND (return_rate_quantity < 0.5 OR return_rate_quantity IS NULL)
       AND stockcode NOT IN ('B', 'C2') 
       AND stockcode NOT IN (
    SELECT DISTINCT stockcode
    FROM mart.retail_enriched
    WHERE customerid IN (
        SELECT DISTINCT customerid
        FROM staging.top_anomalies
        WHERE anomaly_type = 'anomaly_customer'
    )
)
       ORDER BY total_sales desc, order_penetration desc
       """).df()
print(df_niche.head(25)[['StockCode', 'description', 'in_orders', 'total_sales', 'return_rate_quantity', 'order_penetration','return_rate_value']])

  StockCode                         description  in_orders  total_sales  \
0     22655         VINTAGE RED KITCHEN CABINET         38      8125.00   
1     23134     LARGE ZINC HEART WALL ORGANISER         40      6431.11   
2     22826       LOVE SEAT ANTIQUE WHITE METAL         41      6210.00   
3    47556B                TEA TIME TEA TOWELS           3      6045.00   
4     22783    SET 3 WICKER OVAL BASKETS W LIDS         34      5961.25   
5     62018                           SOMBRERO          61      5716.85   
6     23135     SMALL ZINC HEART WALL ORGANISER         47      5416.18   
7     22827  RUSTIC  SEVENTEEN DRAWER SIDEBOARD         27      5415.00   

   return_rate_quantity  order_penetration  return_rate_value  
0              0.166667           0.001833           0.212985  
1                   NaN           0.001930                NaN  
2              0.068966           0.001978           0.083333  
3                   NaN           0.000145                NaN  
4   

# Stars
  StockCode                         description  in_orders  total_sales 

0      48194                       DOORMAT HEARTS        593     22114.42   
1      22189              CREAM HEART CARD HOLDER        524     23435.82   
2      48187                  DOORMAT NEW ENGLAND        600     24108.39   
3      21623         VINTAGE UNION JACK MEMOBOARD        221     22980.24   
4      21137             BLACK RECORD COVER FRAME        375     40633.38   
5      21523   DOORMAT FANCY FONT HOME SWEET HOME        502     21593.74   
6      84879        ASSORTED COLOUR BIRD ORNAMENT       1455     58927.62   
7      23209             LUNCH BAG VINTAGE DOILY        1099     21418.82   
8      21915               RED  HARMONICA IN BOX         662     26294.08   
9      48138                   DOORMAT UNION FLAG        616     25534.15   
10     23084                   RABBIT NIGHT LIGHT       1007     66870.03 

Return_rate_quantity mellan 0.000878 (Doormat Hearts) och 0.003410 (Rabbit Night Light). 25 percentil är sub 0.005761.

Return_rate_value mellan 0.001078 (Doormat Hearts) och 003818 (Doormat Union Flag) 25 percetil är sub 0.005635.

Order_penetration 0.010662 (Vintage Union Jack Memoboard) och 0.070195 (Assorted Colour Bird Ornament). 75 percentil är över 0.0079.

Dessa produkter utmärker sig på samtliga nyckelmått. Returgrad, både i kvantitet och värde, ligger långt under genomsnittet samtidigt som produkterna säljer mycket och brett. 

Bland dessa återfinns ett par produkter från vår top-lista rankat på föräljning. Exempelvis Assorted Colour Bird Ornament, Rabbit Night Light och Black Record Cover Frame.

Det är just dessa som vi vill fortsätta lyfta fram och använda som ankare i exempelvis bundling- eller cross-selling-strategier.


# Risky
  StockCode                         description  in_orders  total_sales

0     23284       DOORMAT KEEP CALM AND COME IN        731     38133.64   
1     22699    ROSES REGENCY TEACUP AND SAUCER        1066     29056.92   
2     23201                  JUMBO BAG ALPHABET        906     27850.83   
3     23199                    JUMBO BAG APPLES        978     30450.69   
4     22423            REGENCY CAKESTAND 3 TIER       1989    174156.54   
5    85123A  WHITE HANGING HEART T-LIGHT HOLDER       2203    104462.75   
6     21843            RED RETROSPOT CAKE STAND        539     21354.30   
7     21175         GIN + TONIC DIET METAL SIGN        809     27029.97   
8     71477   COLOUR GLASS. STAR T-LIGHT HOLDER        320     20650.66   
9     48185                  DOORMAT FAIRY CAKE        365     23872.91

Produkter som vi klassificerar som "Risky" bidrar signifikant till försäljningen, men de uppvisar också svagheter i form av höga returgrader.
Returgraden (kvantitet) ligger mellan 0.040909 (Doormat Keep Calm and Come In) och 0.186445 (Doormat Fairy Cake), med motsvarande returgrader sett till värde mellan 0.038635 (Jumbo Bag Alphabet) och 0.190798 (Doormat Fairy Cake). Detta betyder att trotts betydande intäkter så urholkas försäljningen samtidigt av returer som ligger över genomsnittet.  

Tre av dessa produkter återfinns bland våra top-produkter. Regency Cakestand 3 Tier som uppmärksammades tidigare, White Hanging Heart T-Light Holder och Doormat Keep Calm and Come In. 

Att identifiera och åtgärda den höga returgraden bland dessa produkter skulle i sig leda till både bättre lönsamhet och ökad kundnöjdhet.

# Niche
  StockCode                         description  in_orders  total_sales

0     22655         VINTAGE RED KITCHEN CABINET         38      8125.00   
1     23134     LARGE ZINC HEART WALL ORGANISER         40      6431.11   
2     22826       LOVE SEAT ANTIQUE WHITE METAL         41      6210.00   
3    47556B                TEA TIME TEA TOWELS           3      6045.00   
4     22783    SET 3 WICKER OVAL BASKETS W LIDS         34      5961.25   
5     62018                           SOMBRERO          61      5716.85   
6     23135     SMALL ZINC HEART WALL ORGANISER         47      5416.18   
7     22827  RUSTIC  SEVENTEEN DRAWER SIDEBOARD         27      5415.00 

Segmentet Niche består av produkter som har en relativt god försäljning men som inte når särskilt brett. 

Här finner vi produkter på båda sidorna av extrema returgradsvärden. Ett par produkter har returgrad (värde) uppemot 0.212985 medans andra saknar returer helt.  

De produkter av störst intresse här är de som kombinerar låg order-penetration med avsaknad av returer, såsom Tea Time Tea Towels och Sombrero. Dessa produkter tycks indikera en god produkt/market-fit inom den något smalare kundgruppen.

Genom att lyfta fram dessa produkter för att öka order penetration, genom exempelvis cross-selling och bundling med "Stars", kan vara ett effektivt sätt att driva ytterligare intäkter utan att öka returvolymen. 